# Geographic Analysis of São Paulo Voters' Educational Profile (2022)

**Author:** Mauro Santos

## Summary
* [1. Introduction](#introduction)
* [2. Environment Setup](#setup)
* [3. Data Loading and Processing](#processing)
* [4. Aggregation and Transformation by Municipality](#aggregation)
* [5. Geospatial Data Visualization](#visualization)
* [6. Conclusion and Next Steps](#conclusion)

---

## 1. Introduction <a name="introduction"></a>
This project aims to perform a detailed analysis of the educational profile of voters in the state of São Paulo. Using public data from the [TSE Electoral Data Repository](https://dados.tse.jus.br/dataset/eleitorado-por-secao-eleitoral) for the year 2022, we investigate the geographic distribution of different education levels across the state.

One of the main technical challenges was handling a massive data file (5GB), which required chunk-based processing to ensure computational efficiency without exhausting memory.

The final result consists of static and interactive choropleth maps showing the share of voters by education level in each municipality, enabling clear comparative visual analysis.

**Tools used:** Python, Pandas, GeoPandas, and Matplotlib.

## 2. Environment Setup <a name="setup"></a>
In this section, we perform the initial setup of the working environment. This includes:
* Mounting Google Drive to access data files (CSV and Shapefile).
* Installing required geospatial libraries such as `geopandas`.
* Importing all libraries used throughout the project.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
csv_path = '/data/perfil_eleitor_secao_2022_SP.csv'
shapefile_path = '/data/SP_Municipios_2024.shp'

## 3. Data Loading and Processing <a name="processing"></a>
The original TSE data file is 5GB, which may exceed available RAM in environments such as Google Colab. To work around this limitation, we read the CSV file in chunks.

For each chunk, we immediately filter records for São Paulo state (`SG_UF == 'SP'`) and keep only columns relevant to the analysis. This optimized approach allows us to process the entire dataset efficiently.

In [ ]:
chunk_size = 500000
sp_chunks = []

csv_reader = pd.read_csv(
    csv_path,
    encoding='latin-1',
    sep=';',
    chunksize=chunk_size,
    usecols=['SG_UF', 'NM_MUNICIPIO', 'DS_GRAU_ESCOLARIDADE', 'QT_ELEITORES_PERFIL']  # Keep only useful columns
)

print("Starting CSV reading and filtering...")

for chunk in csv_reader:
    # sp_chunk = chunk[chunk['SG_UF'] == 'SP']
    sp_chunks.append(chunk)

df_sp = pd.concat(sp_chunks)

print(f"Processing finished! Total of {len(df_sp)} records found for SP.")
df_sp.head()

## 4. Aggregation and Transformation by Municipality <a name="aggregation"></a>
With São Paulo data loaded, the next step is to transform it for analysis. We group data by municipality and education level to compute the number of voters in each category.

The key step is calculating the **percentage share**. Instead of using absolute numbers, we compute how much each education level represents within the municipality's own total electorate. This enables fairer comparisons and prevents larger cities from masking patterns in smaller municipalities.

In [ ]:
print("Aggregating data by municipality and education level...")
df_aggregated = df_sp.groupby(['NM_MUNICIPIO', 'DS_GRAU_ESCOLARIDADE'])['QT_ELEITORES_PERFIL'].sum().reset_index()

df_pivot = df_aggregated.pivot(
    index='NM_MUNICIPIO',
    columns='DS_GRAU_ESCOLARIDADE',
    values='QT_ELEITORES_PERFIL'
).reset_index()

df_pivot = df_pivot.fillna(0)
education_columns = df_pivot.columns.drop('NM_MUNICIPIO')
df_pivot['TOTAL_VOTERS'] = df_pivot[education_columns].sum(axis=1)

for column in education_columns:
    df_pivot[f'{column}_PCT'] = (df_pivot[column] / df_pivot['TOTAL_VOTERS']) * 100

print("Data aggregated and proportions calculated successfully!")
df_pivot.head()

In [ ]:
print("Loading shapefile...")

gdf_sp_map = gpd.read_file(shapefile_path)
shp_municipality_name_column = 'NM_MUN'
df_pivot['join_key'] = df_pivot['NM_MUNICIPIO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.upper()
gdf_sp_map['join_key'] = gdf_sp_map[shp_municipality_name_column].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.upper()

final_map = gdf_sp_map.merge(
    df_pivot,
    on='join_key',
    how='left'
)

print("Shapefile loaded and merged with voter data.")
final_map.head()

In [ ]:
column_to_plot = 'SUPERIOR COMPLETO_PCT'

if column_to_plot not in final_map.columns:
    print(f"Error: Column '{column_to_plot}' was not found.")
    print("Check whether Block 4 was executed correctly.")
else:
    fig, ax = plt.subplots(1, 1, figsize=(15, 15))

    final_map.plot(
        column=column_to_plot,
        cmap='viridis',
        linewidth=0.5,
        ax=ax,
        edgecolor='0.9',
        legend=True,
        scheme='quantiles',
        k=7,
        legend_kwds={'title': "% of Voters with Completed Higher Education"}
    )

    ax.axis('off')
    ax.set_title('Percentage of Voters with Completed Higher Education by Municipality in SP', fontdict={'fontsize': '16', 'fontweight': '3'})
    plt.show()

## 5. Geospatial Data Visualization <a name="visualization"></a>
This is the core stage of the analysis. We merge aggregated electoral data with the municipalities Shapefile to generate geospatial visualizations.

In [ ]:
columns_to_plot = [
    'ANALFABETO_PCT',
    'ENSINO FUNDAMENTAL INCOMPLETO_PCT',
    'ENSINO MÉDIO COMPLETO_PCT',
    'SUPERIOR COMPLETO_PCT'
]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 20))
axes = axes.flatten()

for i, column in enumerate(columns_to_plot):
    ax = axes[i]
    final_map.plot(
        column=column,
        ax=ax,
        legend=True,
        cmap='viridis',
        scheme='quantiles',
        k=5,
        legend_kwds={'title': '%'}
    )
    ax.set_title(column.replace('_PCT', ''), fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Conclusion <a name="conclusion"></a>

This analysis suggests that the southern region of the state has a higher concentration of municipalities with illiterate voters and, in contrast, fewer voters with completed higher education. It also indicates that municipalities near the São Paulo metropolitan region tend to show higher education levels, especially in the Vale do Paraíba and central-west regions of the state.